# Rule-Based Baseline for Social Correction

This notebook implements a simple rule-based rewriting baseline for improving the social appropriateness of chatbot responses. The system uses deterministic rewrite rules based on the project’s identified social failure categories, such as harsh tone, overconfidence, lack of acknowledgment, and overly direct phrasing.

The goal of this baseline is to provide an interpretable early comparison point before supervised fine-tuning.

In [3]:
import json
import re
from pathlib import Path
import random
import pandas as pd

In [4]:
data_path = Path("../data/correction_dataset.json")

with open(data_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} examples.")
dataset[:2]

Loaded 15 examples.


[{'user': 'Can you check if my answer is correct?',
  'bad': 'That’s wrong.',
  'better': 'I see what you’re trying to do, but there may be an issue in your solution. Let’s go through it together.',
  'failure_type': 'Overly Harsh or Judgmental Language'},
 {'user': 'Why isn’t my code working?',
  'bad': 'Your code is bad.',
  'better': 'I think there might be a small issue in your code. Let’s take a closer look to figure out what’s going wrong.',
  'failure_type': 'Negative Framing Instead of Constructive Framing'}]

In [5]:
df = pd.DataFrame(dataset)
df

,user,bad,better,failure_type
0,Can you check if my answer is correct?,That’s wrong.,"I see what you’re trying to do, but there may ...",Overly Harsh or Judgmental Language
1,Why isn’t my code working?,Your code is bad.,I think there might be a small issue in your c...,Negative Framing Instead of Constructive Framing
2,Is this definitely the right answer?,"Yes, this is definitely correct.","This looks correct to me, but it might be wort...",Overconfidence / Lack of Uncertainty
3,I’m confused about this concept.,"It’s simple, you’re overthinking it.",I understand why this might feel confusing—let...,Lack of Acknowledgment
4,Can you explain this again?,I already explained it.,Of course! Let me try explaining it a differen...,Emotional Mismatch (Too Cold or Abrupt)
5,What should I do to fix this?,Just redo it.,You might want to try revising this part and c...,Direct or Commanding Tone
6,I think my answer makes sense.,It doesn’t.,"I see your reasoning, but there might be a sma...",Overly Harsh or Judgmental Language
7,This result seems strange.,That’s because you did it wrong.,That result does look unusual—there might be a...,Negative Framing Instead of Constructive Framing
8,Can you help me understand this?,Just read the instructions.,Sure! Let’s go through the instructions togeth...,Emotional Mismatch (Too Cold or Abrupt)
9,Is this the best way to solve it?,"Yes, this is the best way.","This is a good approach, though there may be o...",Overconfidence / Lack of Uncertainty


## Rule Design

The rule-based baseline is designed around the project's social failure taxonomy. The rules are intentionally simple and interpretable rather than context-sensitive or learned.

The baseline focuses on four main strategies:

1. Replacing harsh or judgmental wording with softer alternatives
2. Reducing overconfident language through hedging
3. Adding acknowledgment phrases to abrupt responses
4. Reframing direct commands as collaborative suggestions

These rules do not fully model conversational nuance, but they provide a concrete and explainable baseline for comparison.

In [6]:
ACKNOWLEDGMENT_PHRASES = [
    "I see what you mean.",
    "I understand why that might be frustrating.",
    "Thanks for pointing that out.",
    "I see your reasoning.",
    "I understand why that could be confusing."
]

HEDGING_REPLACEMENTS = {
    "definitely": "likely",
    "obviously": "it seems",
    "clearly": "it appears",
    "always": "often",
    "never": "may not",
    "best way": "a good way"
}

HARSH_REPLACEMENTS = {
    "wrong": "may not be correct",
    "bad": "could be improved",
    "dumb": "not very effective",
    "horrible": "could be improved",
    "stupid": "not a strong approach"
}

FULL_SENTENCE_REPLACEMENTS = {
    "That’s wrong.": "I see what you’re trying to do, but there may be an issue here.",
    "That's wrong.": "I see what you’re trying to do, but there may be an issue here.",
    "Your code is bad.": "I think there might be a small issue in your code. Let’s take a closer look.",
    "It doesn’t.": "I see your reasoning, but there may be a small mistake.",
    "You didn’t.": "You’re on the right track, but there may be a small error.",
    "Just redo it.": "You might want to revise it and try again step by step.",
    "Fix everything.": "There are a few important areas to improve, and we can work through them together.",
    "Just read the instructions.": "Let’s go through the instructions together and break them down.",
    "I already explained it.": "Of course — let me explain it again in a different way.",
    "That’s not a good idea.": "I see what you’re aiming for, though this approach may have some challenges."
}

In [7]:
def replace_case_insensitive(text, old, new):
    pattern = re.compile(rf"\b{re.escape(old)}\b", re.IGNORECASE)
    return pattern.sub(new, text)

def starts_abruptly(text):
    abrupt_starts = [
        "that’s", "that's", "you didn’t", "you didn't", "it doesn’t", "it doesn't",
        "just", "no", "wrong", "bad"
    ]
    text_lower = text.strip().lower()
    return any(text_lower.startswith(start) for start in abrupt_starts)

In [8]:
def rewrite_response(text):
    rewritten = text.strip()

    # 1. Full sentence replacements for especially blunt responses
    if rewritten in FULL_SENTENCE_REPLACEMENTS:
        return FULL_SENTENCE_REPLACEMENTS[rewritten]

    # 2. Replace harsh words with softer alternatives
    for harsh_word, soft_word in HARSH_REPLACEMENTS.items():
        rewritten = replace_case_insensitive(rewritten, harsh_word, soft_word)

    # 3. Replace overconfident terms with hedged language
    for strong_word, hedged_word in HEDGING_REPLACEMENTS.items():
        rewritten = replace_case_insensitive(rewritten, strong_word, hedged_word)

    # 4. Add acknowledgment to short or abrupt responses
    if len(rewritten.split()) <= 6 or starts_abruptly(rewritten):
        ack = random.choice(ACKNOWLEDGMENT_PHRASES)
        rewritten = f"{ack} {rewritten}"

    # 5. Reframe certain direct phrases into collaborative suggestions
    if rewritten.lower().startswith("you should"):
        rewritten = rewritten.replace("You should", "You might consider")
        rewritten = rewritten.replace("you should", "you might consider")

    if rewritten.lower().startswith("do this"):
        rewritten = rewritten.replace("Do this", "You could try this")
        rewritten = rewritten.replace("do this", "you could try this")

    return rewritten

In [9]:
test_examples = [
    "That’s wrong.",
    "Your code is bad.",
    "Yes, this is definitely correct.",
    "Just redo it.",
    "I already explained it."
]

for example in test_examples:
    print("Original:", example)
    print("Rewritten:", rewrite_response(example))
    print("-" * 60)

Original: That’s wrong.
Rewritten: I see what you’re trying to do, but there may be an issue here.
------------------------------------------------------------
Original: Your code is bad.
Rewritten: I think there might be a small issue in your code. Let’s take a closer look.
------------------------------------------------------------
Original: Yes, this is definitely correct.
Rewritten: I see your reasoning. Yes, this is likely correct.
------------------------------------------------------------
Original: Just redo it.
Rewritten: You might want to revise it and try again step by step.
------------------------------------------------------------
Original: I already explained it.
Rewritten: Of course — let me explain it again in a different way.
------------------------------------------------------------


In [10]:
results = []

for item in dataset:
    rewritten = rewrite_response(item["bad"])
    results.append({
        "user": item["user"],
        "bad": item["bad"],
        "rule_based_output": rewritten,
        "target_better": item["better"],
        "failure_type": item["failure_type"]
    })

results_df = pd.DataFrame(results)
results_df

,user,bad,rule_based_output,target_better,failure_type
0,Can you check if my answer is correct?,That’s wrong.,"I see what you’re trying to do, but there may ...","I see what you’re trying to do, but there may ...",Overly Harsh or Judgmental Language
1,Why isn’t my code working?,Your code is bad.,I think there might be a small issue in your c...,I think there might be a small issue in your c...,Negative Framing Instead of Constructive Framing
2,Is this definitely the right answer?,"Yes, this is definitely correct.",I understand why that might be frustrating. Ye...,"This looks correct to me, but it might be wort...",Overconfidence / Lack of Uncertainty
3,I’m confused about this concept.,"It’s simple, you’re overthinking it.","I see your reasoning. It’s simple, you’re over...",I understand why this might feel confusing—let...,Lack of Acknowledgment
4,Can you explain this again?,I already explained it.,Of course — let me explain it again in a diffe...,Of course! Let me try explaining it a differen...,Emotional Mismatch (Too Cold or Abrupt)
5,What should I do to fix this?,Just redo it.,You might want to revise it and try again step...,You might want to try revising this part and c...,Direct or Commanding Tone
6,I think my answer makes sense.,It doesn’t.,"I see your reasoning, but there may be a small...","I see your reasoning, but there might be a sma...",Overly Harsh or Judgmental Language
7,This result seems strange.,That’s because you did it wrong.,I understand why that could be confusing. That...,That result does look unusual—there might be a...,Negative Framing Instead of Constructive Framing
8,Can you help me understand this?,Just read the instructions.,Let’s go through the instructions together and...,Sure! Let’s go through the instructions togeth...,Emotional Mismatch (Too Cold or Abrupt)
9,Is this the best way to solve it?,"Yes, this is the best way.","Yes, this is the a good way.","This is a good approach, though there may be o...",Overconfidence / Lack of Uncertainty


In [11]:
for _, row in results_df.iterrows():
    print("USER:", row["user"])
    print("BAD RESPONSE:", row["bad"])
    print("RULE-BASED OUTPUT:", row["rule_based_output"])
    print("TARGET BETTER RESPONSE:", row["target_better"])
    print("FAILURE TYPE:", row["failure_type"])
    print("-" * 90)

USER: Can you check if my answer is correct?
BAD RESPONSE: That’s wrong.
RULE-BASED OUTPUT: I see what you’re trying to do, but there may be an issue here.
TARGET BETTER RESPONSE: I see what you’re trying to do, but there may be an issue in your solution. Let’s go through it together.
FAILURE TYPE: Overly Harsh or Judgmental Language
------------------------------------------------------------------------------------------
USER: Why isn’t my code working?
BAD RESPONSE: Your code is bad.
RULE-BASED OUTPUT: I think there might be a small issue in your code. Let’s take a closer look.
TARGET BETTER RESPONSE: I think there might be a small issue in your code. Let’s take a closer look to figure out what’s going wrong.
FAILURE TYPE: Negative Framing Instead of Constructive Framing
------------------------------------------------------------------------------------------
USER: Is this definitely the right answer?
BAD RESPONSE: Yes, this is definitely correct.
RULE-BASED OUTPUT: I understand wh

In [12]:
def analyze_changes(original, rewritten):
    notes = []

    if original != rewritten:
        notes.append("changed_output")

    if any(phrase in rewritten for phrase in ACKNOWLEDGMENT_PHRASES):
        notes.append("added_acknowledgment")

    if any(word in rewritten.lower() for word in ["likely", "seems", "appears", "might", "could"]):
        notes.append("softened_language")

    if any(word in rewritten.lower() for word in ["together", "consider", "try", "improve"]):
        notes.append("more_constructive")

    return notes

analysis_rows = []

for row in results:
    analysis_rows.append({
        "bad": row["bad"],
        "rule_based_output": row["rule_based_output"],
        "failure_type": row["failure_type"],
        "notes": ", ".join(analyze_changes(row["bad"], row["rule_based_output"]))
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df

,bad,rule_based_output,failure_type,notes
0,That’s wrong.,"I see what you’re trying to do, but there may ...",Overly Harsh or Judgmental Language,"changed_output, more_constructive"
1,Your code is bad.,I think there might be a small issue in your c...,Negative Framing Instead of Constructive Framing,"changed_output, softened_language"
2,"Yes, this is definitely correct.",I understand why that might be frustrating. Ye...,Overconfidence / Lack of Uncertainty,"changed_output, added_acknowledgment, softened..."
3,"It’s simple, you’re overthinking it.","I see your reasoning. It’s simple, you’re over...",Lack of Acknowledgment,"changed_output, added_acknowledgment"
4,I already explained it.,Of course — let me explain it again in a diffe...,Emotional Mismatch (Too Cold or Abrupt),changed_output
5,Just redo it.,You might want to revise it and try again step...,Direct or Commanding Tone,"changed_output, softened_language, more_constr..."
6,It doesn’t.,"I see your reasoning, but there may be a small...",Overly Harsh or Judgmental Language,changed_output
7,That’s because you did it wrong.,I understand why that could be confusing. That...,Negative Framing Instead of Constructive Framing,"changed_output, added_acknowledgment, softened..."
8,Just read the instructions.,Let’s go through the instructions together and...,Emotional Mismatch (Too Cold or Abrupt),"changed_output, more_constructive"
9,"Yes, this is the best way.","Yes, this is the a good way.",Overconfidence / Lack of Uncertainty,changed_output


In [13]:
changed_count = sum(1 for row in results if row["bad"] != row["rule_based_output"])
ack_count = sum(1 for row in results if any(phrase in row["rule_based_output"] for phrase in ACKNOWLEDGMENT_PHRASES))
softened_count = sum(
    1 for row in results
    if any(word in row["rule_based_output"].lower() for word in ["likely", "might", "could", "appears", "seems"])
)

print(f"Number of rewritten responses changed: {changed_count}/{len(results)}")
print(f"Number of responses with acknowledgment added: {ack_count}/{len(results)}")
print(f"Number of responses with softened language: {softened_count}/{len(results)}")

Number of rewritten responses changed: 15/15
Number of responses with acknowledgment added: 4/15
Number of responses with softened language: 5/15


## Early Observations and Limitations

The rule-based baseline successfully softens many blunt or overconfident responses and introduces more collaborative phrasing in several cases. This makes it useful as an interpretable first-stage baseline.

However, the system also has clear limitations. Because the rules are hand-written and context-independent, some rewrites may sound repetitive, awkward, or insufficiently tailored to the user’s prompt. The baseline also cannot reason deeply about social context, user emotion, or more subtle pragmatic failures. These limitations motivate the next stage of the project: learning corrections from data through supervised fine-tuning.

In [14]:
output_path = Path("../data/rule_based_results.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved rule-based results to {output_path}")

Saved rule-based results to ../data/rule_based_results.json


## Lightweight Evaluation Setup

To evaluate the rule-based baseline, we use a simple heuristic scoring framework. Rather than measuring only exact match to the target correction, we check whether the rewritten response shows socially important behaviors such as acknowledgment, constructive phrasing, reduced harshness, and reduced overconfidence. We also include a simple fluency flag to capture awkward or unnatural outputs.

In [15]:
ACK_WORDS = [
    "i see", "i understand", "thanks for", "you're on the right track",
    "let's", "of course"
]

CONSTRUCTIVE_WORDS = [
    "improve", "together", "try", "consider", "help", "go through",
    "take a closer look", "work through", "step by step"
]

HEDGE_WORDS = [
    "might", "may", "could", "likely", "appears", "seems"
]

HARSH_WORDS = [
    "wrong", "bad", "stupid", "dumb", "horrible"
]

AWKWARD_PATTERNS = [
    "the a ",
    "may not be correct every time",
    "it’s simple, you’re overthinking it",
]

In [16]:
def evaluate_response(text):
    text_lower = text.lower()

    scores = {
        "acknowledgment": int(any(word in text_lower for word in ACK_WORDS)),
        "constructive": int(any(word in text_lower for word in CONSTRUCTIVE_WORDS)),
        "hedging": int(any(word in text_lower for word in HEDGE_WORDS)),
        "contains_harsh_word": int(any(word in text_lower for word in HARSH_WORDS)),
        "awkward_or_limited": int(any(pattern in text_lower for pattern in AWKWARD_PATTERNS))
    }

    return scores

In [17]:
eval_rows = []

for row in results:
    scores = evaluate_response(row["rule_based_output"])
    eval_rows.append({
        "user": row["user"],
        "bad": row["bad"],
        "rule_based_output": row["rule_based_output"],
        "failure_type": row["failure_type"],
        **scores
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

,user,bad,rule_based_output,failure_type,acknowledgment,constructive,hedging,contains_harsh_word,awkward_or_limited
0,Can you check if my answer is correct?,That’s wrong.,"I see what you’re trying to do, but there may ...",Overly Harsh or Judgmental Language,1,1,1,0,0
1,Why isn’t my code working?,Your code is bad.,I think there might be a small issue in your c...,Negative Framing Instead of Constructive Framing,0,1,1,0,0
2,Is this definitely the right answer?,"Yes, this is definitely correct.",I understand why that might be frustrating. Ye...,Overconfidence / Lack of Uncertainty,1,0,1,0,0
3,I’m confused about this concept.,"It’s simple, you’re overthinking it.","I see your reasoning. It’s simple, you’re over...",Lack of Acknowledgment,1,0,0,0,1
4,Can you explain this again?,I already explained it.,Of course — let me explain it again in a diffe...,Emotional Mismatch (Too Cold or Abrupt),1,0,0,0,0
5,What should I do to fix this?,Just redo it.,You might want to revise it and try again step...,Direct or Commanding Tone,0,1,1,0,0
6,I think my answer makes sense.,It doesn’t.,"I see your reasoning, but there may be a small...",Overly Harsh or Judgmental Language,1,0,1,0,0
7,This result seems strange.,That’s because you did it wrong.,I understand why that could be confusing. That...,Negative Framing Instead of Constructive Framing,1,0,1,0,0
8,Can you help me understand this?,Just read the instructions.,Let’s go through the instructions together and...,Emotional Mismatch (Too Cold or Abrupt),0,1,0,0,0
9,Is this the best way to solve it?,"Yes, this is the best way.","Yes, this is the a good way.",Overconfidence / Lack of Uncertainty,0,0,0,0,1


In [18]:
summary = {
    "total_examples": len(eval_df),
    "acknowledgment_count": int(eval_df["acknowledgment"].sum()),
    "constructive_count": int(eval_df["constructive"].sum()),
    "hedging_count": int(eval_df["hedging"].sum()),
    "remaining_harsh_count": int(eval_df["contains_harsh_word"].sum()),
    "awkward_or_limited_count": int(eval_df["awkward_or_limited"].sum())
}

summary

{'total_examples': 15,
 'acknowledgment_count': 8,
 'constructive_count': 6,
 'hedging_count': 10,
 'remaining_harsh_count': 0,
 'awkward_or_limited_count': 3}

In [19]:
for key, value in summary.items():
    print(f"{key}: {value}")

total_examples: 15
acknowledgment_count: 8
constructive_count: 6
hedging_count: 10
remaining_harsh_count: 0
awkward_or_limited_count: 3
